# DPR V4 — A100 Colab Experiment

**Operator:** Open in Google Colab, select an **NVIDIA A100 GPU**, and run cells in order. You must authorize the Google Drive mount. This notebook **refuses to run without a mounted Drive**, so completed checkpoints and metrics are not lost silently.

**Study:** 10,000 unique training questions, 15,000 mining passages, 4 explicit model-hard negatives versus 32 in-batch negatives, batch size 33, 300 optimizer steps per strategy, seed 42. This is a scaled single-seed experiment; it does **not** establish statistical significance. SQuAD one-question-per-context preprocessing means all numbers must be verified in the dataset check.

**Important:** Upload `src/train_v4.py` to **your** GitHub repository before your brother runs this notebook. All other modules retain their V3 interfaces. Run outputs go to the connected **Google Drive** (not the GitHub repo).

## 1. A100 and Google Drive — required

In [ ]:
import os, sys, subprocess, json, time, shutil, hashlib, shlex
from pathlib import Path
import torch
assert torch.cuda.is_available(), "Enable GPU runtime in Colab settings"
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print("GPU:", GPU_NAME, "| VRAM:", round(VRAM_GB, 1), "GB")
assert "A100" in GPU_NAME, "An A100 was requested; stopping rather than silently using another GPU."
# headless run on probook: no Drive mount, outputs go to a local folder that colab-job fetches
DRIVE_ROOT = Path('/content/dpr_v4_a100_seed42')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
probe = DRIVE_ROOT / '.write_test'
probe.write_text('drive-mounted')
assert probe.read_text() == 'drive-mounted'
probe.unlink()
print('Confirmed persistent Drive output:', DRIVE_ROOT)


GPU: NVIDIA A100-SXM4-40GB | VRAM: 39.5 GB
Confirmed persistent Drive output: /content/dpr_v4_a100_seed42


## 2. Clone the source and install dependencies

In [ ]:
REPO_URL = 'https://github.com/utsab818/dpr-mining-study.git'
REPO = Path('/content/dpr-mining-study')
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Source commit:', COMMIT)
for filename in ['data.py', 'model.py', 'loss.py', 'warmup_v3.py', 'mine_negatives.py', 'train_v4.py', 'evaluate.py']:
    assert (REPO/'src'/filename).is_file(), f'Missing src/{filename}; push the V4 trainer to GitHub first.'
commit_file = DRIVE_ROOT / 'source_commit.txt'
if commit_file.exists():
    assert commit_file.read_text().strip() == COMMIT, 'This Drive experiment folder was created for a different Git commit. Use a new folder.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.35,<5', 'datasets>=2.16,<5', 'safetensors'], check=True)
import datasets, transformers
commit_file.write_text(COMMIT+'\n')
(DRIVE_ROOT/'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'], text=True))
print('torch:',torch.__version__,'transformers:',transformers.__version__,'datasets:',datasets.__version__)


Source commit: 105bb0e99a9e7cfd03db2cb64bdaa827d21ec4f9


torch: 2.11.0+cu130 transformers: 4.57.6 datasets: 4.8.5


## 3. Verify available SQuAD passages before using the A100

SQuAD v1.1 has 87,599 training QA rows but far fewer *unique contexts*. Our dataset class keeps only one question per unique context. We therefore choose **15,000** mining passages instead of 20,000, and verify that the class produces the requested training and mining sizes. <https://huggingface.co/datasets/rajpurkar/squad>

In [ ]:
subprocess.run([sys.executable, '-c', 'from src.data import DPRDataset\nn_train, n_passages = 10000, 15000\ndata = DPRDataset(split="train", max_samples=n_passages)\nassert len(data.examples) == n_passages, (len(data.examples), n_passages)\nassert len({e["passage"] for e in data.examples}) == n_passages\nassert all(e["question"] and e["passage"] for e in data.examples[:n_train])\nprint("Dataset verified:", n_train, "training queries in", n_passages, "unique passages")'], cwd=REPO, check=True)


CompletedProcess(args=['/usr/bin/python3', '-c', 'from src.data import DPRDataset\nn_train, n_passages = 10000, 15000\ndata = DPRDataset(split="train", max_samples=n_passages)\nassert len(data.examples) == n_passages, (len(data.examples), n_passages)\nassert len({e["passage"] for e in data.examples}) == n_passages\nassert all(e["question"] and e["passage"] for e in data.examples[:n_train])\nprint("Dataset verified:", n_train, "training queries in", n_passages, "unique passages")'], returncode=0)

## 4. Stage runner — checkpoint to Google Drive after each successful stage

In [ ]:
def run_stage(stage_name, args, artifacts):
    stage_dir = DRIVE_ROOT / stage_name
    stage_dir.mkdir(parents=True, exist_ok=True)
    local = [REPO / p for p in artifacts]
    saved = [stage_dir / p for p in artifacts]
    complete = stage_dir / 'completed.json'
    if complete.is_file() and all(p.is_file() for p in saved):
        info = json.loads(complete.read_text())
        assert info['git_commit'] == COMMIT and info['args'] == args, 'Stage arguments or source changed; start a new experiment folder.'
        for dest, src in zip(local, saved):
            dest.parent.mkdir(parents=True, exist_ok=True)
            if not dest.exists(): shutil.copy2(src, dest)
        print('RESTORED:', stage_name)
        return
    if complete.exists() or any(p.exists() for p in saved):
        raise RuntimeError(f'Incomplete stage {stage_name}: inspect Drive outputs and use a fresh run folder if needed.')
    for p in local:
        if p.exists(): raise FileExistsError(f'Local path already exists: {p}')
    print('RUN:',stage_name,shlex.join(args),flush=True)
    t0=time.perf_counter()
    with (stage_dir/'stdout.log').open('w') as log:
        p=subprocess.Popen([sys.executable,'-u','-m',*args], cwd=REPO,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line,end='')
            log.write(line);log.flush()
        rc=p.wait()
    if rc: raise RuntimeError(f'Stage {stage_name} failed; see {stage_dir}/stdout.log')
    for src,dest in zip(local,saved):
        assert src.is_file(),f'Missing stage output: {src}'
        dest.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(src,dest)
        assert dest.stat().st_size == src.stat().st_size
    complete.write_text(json.dumps({'name':stage_name,'git_commit':COMMIT,'args':args,
                                    'seconds':round(time.perf_counter()-t0,2),'artifacts':artifacts},indent=2))
    print('PERSISTED:',stage_name,'at',stage_dir)


## 5. Warm-up — 10,000 questions, 2 epochs

In [ ]:
WARM_DIR = 'experiments/v4/warmup/full'
WARM_BASE = 'dpr_warmup_n10000_b32_seed42_steps624'
WARM_PT = f'{WARM_DIR}/{WARM_BASE}.pt'
run_stage('01_warmup', ['src.warmup_v3','--max-samples','10000','--batch-size','32','--epochs','2',
                        '--seed','42','--output-dir',WARM_DIR],
          [f'{WARM_DIR}/{WARM_BASE}{ext}' for ext in ('.pt','.json','_steps.csv')])


RUN: 01_warmup src.warmup_v3 --max-samples 10000 --batch-size 32 --epochs 2 --seed 42 --output-dir experiments/v4/warmup/full


Device: cuda


Unique question/passage pairs: 10000
Steps per full epoch: 312
Optimizer steps: 624


I0000 00:00:1792935812.563627    5933 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792935812.634176    5933 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1792935815.074815    5933 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792935815.075395    5933 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


...

Step 610 | Scores (32, 32) | Loss 0.0207 | Elapsed 85.9s


Step 620 | Scores (32, 32) | Loss 0.0492 | Elapsed 87.3s


Step 624 | Scores (32, 32) | Loss 0.0042 | Elapsed 87.9s


Average loss: 0.1529503682990113
Training seconds: 87.87143440900013
CUDA peak allocated GB: 3.8428750038146973
Saved: experiments/v4/warmup/full/dpr_warmup_n10000_b32_seed42_steps624.pt
Metadata: experiments/v4/warmup/full/dpr_warmup_n10000_b32_seed42_steps624.json
Step log: experiments/v4/warmup/full/dpr_warmup_n10000_b32_seed42_steps624_steps.csv


PERSISTED: 01_warmup at /content/dpr_v4_a100_seed42/01_warmup


## 6. Warm-up benchmark — compare baseline on 1,000 validation passages

In [ ]:
WARM_EVAL_DIR='experiments/v4/evaluation/warmup'
WARM_EVAL=f'{WARM_EVAL_DIR}/warmup_seed42_1000.csv'
run_stage('01b_warmup_eval', ['src.evaluate','--checkpoint',f'warmup={WARM_PT}',
                             '--max-samples','1000','--batch-size','16','--output',WARM_EVAL],
          [WARM_EVAL, f'{WARM_EVAL_DIR}/warmup_seed42_1000_metadata.json'])
import pandas as pd
display(pd.read_csv(REPO/WARM_EVAL))


RUN: 01b_warmup_eval src.evaluate --checkpoint warmup=experiments/v4/warmup/full/dpr_warmup_n10000_b32_seed42_steps624.pt --max-samples 1000 --batch-size 16 --output experiments/v4/evaluation/warmup/warmup_seed42_1000.csv


Device: cuda


Validation pairs: 1000



===== WARMUP =====


I0000 00:00:1792935923.702058    6506 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792935923.771272    6506 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1792935926.138290    6506 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792935926.138839    6506 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Similarity matrix shape: (1000, 1000)
recall@1: 0.6370
recall@5: 0.8920
recall@20: 0.9750
mrr@10: 0.7474

Results saved to experiments/v4/evaluation/warmup/warmup_seed42_1000.csv
Metadata saved to experiments/v4/evaluation/warmup/warmup_seed42_1000_metadata.json


PERSISTED: 01b_warmup_eval at /content/dpr_v4_a100_seed42/01b_warmup_eval


,strategy,checkpoint,recall@1,recall@5,recall@20,mrr@10
0,warmup,experiments/v4/warmup/full/dpr_warmup_n10000_b...,0.637,0.892,0.975,0.747431


## 7. Mine 4 hard negatives for 10,000 questions from 15,000 passages

In [ ]:
MINE_DIR='experiments/v4/mining/full'
MINE_PT=f'{MINE_DIR}/mined_negatives.pt'
run_stage('02_mining',['src.mine_negatives','--num-questions','10000','--num-passages','15000',
                       '--num-hard','4','--batch-size','32','--seed','42',
                       '--checkpoint',WARM_PT,'--output-dir',MINE_DIR],
          [MINE_PT,f'{MINE_DIR}/passage_index.pt',f'{MINE_DIR}/manifest.json'])
mined=torch.load(REPO/MINE_PT,map_location='cpu',weights_only=True)
ids=mined['hard_negative_indices']
assert len(mined['questions'])==10000 and len(mined['passages'])==15000
assert ids.shape==(10000,4)
assert bool(((ids>=0)&(ids<15000)).all())
assert not bool((ids==torch.arange(10000).unsqueeze(1)).any())
assert all(len(set(row))==4 for row in ids.tolist())
print('Mining positive-ID and duplicate checks passed')


RUN: 02_mining src.mine_negatives --num-questions 10000 --num-passages 15000 --num-hard 4 --batch-size 32 --seed 42 --checkpoint experiments/v4/warmup/full/dpr_warmup_n10000_b32_seed42_steps624.pt --output-dir experiments/v4/mining/full


Device: cuda
Questions: 10000 Passages: 15000


I0000 00:00:1792935948.695655    6662 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792935948.765435    6662 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1792935950.962100    6662 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792935950.962628    6662 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Encoded passages: 32/15000


...

Encoded questions: 6400/10000
Encoded questions: 7200/10000


Encoded questions: 8000/10000
Encoded questions: 8800/10000


Encoded questions: 9600/10000
Passage embeddings: (15000, 768)
Question embeddings: (10000, 768)


Hard-negative indices: (10000, 4)
Diagnostics: {
  "mean_positive_rank": 4.878900051116943,
  "median_positive_rank": 1.0,
  "positive_rank_1_count": 6022,
  "negative_margin_count": 3978,
  "mean_positive_negative_margin": 0.7438905239105225,
  "recall_at_1": 0.6021999716758728,
  "recall_at_5": 0.8514000177383423,
  "recall_at_20": 0.9562000036239624,
  "mean_answer_rejections_per_query": 0.3642,
  "total_answer_rejections": 3642,
  "queries_with_short_answers_len_le_2": 182,
  "max_candidates_scanned_for_one_query": 48
}

Example question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Positive passage: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper sta
Hard negative 1 (passage 6929, score 54.883): Mary had been venerated since Early Christianity, and is considered by millions to be the most meritorious saint o

PERSISTED: 02_mining at /content/dpr_v4_a100_seed42/02_mining
Mining positive-ID and duplicate checks passed


## 8. Train four models — 300 updates, batch size 33, seed 42

All branches start from the **same warm-up checkpoint** and process the same query batches. Random negatives are sampled lazily in `train_v4.py` to avoid precomputing 150 million query–passage eligibility entries. Mixed uses 4 hard + 32 in-batch negatives, not just 4. Passage-encoder compute differs across strategies.

In [ ]:
TRAIN_DIR='experiments/v4/training/full'
CHECKPOINTS={}
for strategy in ['random','hard','mixed','inbatch']:
    k=32 if strategy=='inbatch' else 4
    basename=f'dpr_{strategy}_k{k}_b33_seed42_steps300'
    args=['src.train_v4','--strategy',strategy,'--mined-path',MINE_PT,
          '--warmup-path',WARM_PT,'--output-dir',TRAIN_DIR,
          '--batch-size','33','--num-negatives',str(k),
          '--seed','42','--max-steps','300','--log-every','50']
    run_stage('03_train_'+strategy,args,[f'{TRAIN_DIR}/{basename}{ext}' for ext in ('.pt','.json','_steps.csv')])
    CHECKPOINTS[strategy]=f'{TRAIN_DIR}/{basename}.pt'
print('Checkpoints:',CHECKPOINTS)


RUN: 03_train_random src.train_v4 --strategy random --mined-path experiments/v4/mining/full/mined_negatives.pt --warmup-path experiments/v4/warmup/full/dpr_warmup_n10000_b32_seed42_steps624.pt --output-dir experiments/v4/training/full --batch-size 33 --num-negatives 4 --seed 42 --max-steps 300 --log-every 50


Device: cuda
GPU: NVIDIA A100-SXM4-40GB
Strategy: random Seed: 42


I0000 00:00:1792936179.735234    7681 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792936179.805196    7681 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1792936181.968208    7681 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792936181.968729    7681 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Step 1 | Scores (33, 5) | Loss 0.0007 | Elapsed 0.9s


...

Step 200 | Scores (33, 33) | Loss 0.0943 | Elapsed 28.5s


Step 250 | Scores (33, 33) | Loss 0.0069 | Elapsed 35.5s


Step 300 | Scores (33, 33) | Loss 0.0013 | Elapsed 42.6s


Total optimizer steps: 300
Query exposures: 9900
Passage encoder inputs: 9900
Average loss: 0.03923430117486229
Training seconds: 42.550750634999986
CUDA peak allocated GB: 3.914
Saved: experiments/v4/training/full/dpr_inbatch_k32_b33_seed42_steps300.pt
Metrics: experiments/v4/training/full/dpr_inbatch_k32_b33_seed42_steps300.json
Per-step log: experiments/v4/training/full/dpr_inbatch_k32_b33_seed42_steps300_steps.csv


PERSISTED: 03_train_inbatch at /content/dpr_v4_a100_seed42/03_train_inbatch
Checkpoints: {'random': 'experiments/v4/training/full/dpr_random_k4_b33_seed42_steps300.pt', 'hard': 'experiments/v4/training/full/dpr_hard_k4_b33_seed42_steps300.pt', 'mixed': 'experiments/v4/training/full/dpr_mixed_k4_b33_seed42_steps300.pt', 'inbatch': 'experiments/v4/training/full/dpr_inbatch_k32_b33_seed42_steps300.pt'}


## 9. Evaluate on 1,000 and 2,000 held-out passages

We deliberately use 2,000 (not 5,000) for the larger SQuAD validation evaluation because the one-context-per-positive deduplication produces only about two thousand validation contexts. Both evaluations use the same held-out validation split.

In [ ]:
for nval in [1000,2000]:
    EVAL_DIR=f'experiments/v4/evaluation/validation_{nval}'
    EVAL_OUT=f'{EVAL_DIR}/seed42_{nval}.csv'
    specs=[f'warmup={WARM_PT}']+[f'{name}={path}' for name,path in CHECKPOINTS.items()]
    args=['src.evaluate']
    for spec in specs: args+=['--checkpoint',spec]
    args+=['--max-samples',str(nval),'--batch-size','16','--save-query-ranks','--output',EVAL_OUT]
    run_stage(f'04_evaluation_{nval}',args,[EVAL_OUT,
         f'{EVAL_DIR}/seed42_{nval}_query_ranks.csv',
         f'{EVAL_DIR}/seed42_{nval}_metadata.json'])
    print('VALIDATION SIZE',nval)
    display(pd.read_csv(REPO/EVAL_OUT))


RUN: 04_evaluation_1000 src.evaluate --checkpoint warmup=experiments/v4/warmup/full/dpr_warmup_n10000_b32_seed42_steps624.pt --checkpoint random=experiments/v4/training/full/dpr_random_k4_b33_seed42_steps300.pt --checkpoint hard=experiments/v4/training/full/dpr_hard_k4_b33_seed42_steps300.pt --checkpoint mixed=experiments/v4/training/full/dpr_mixed_k4_b33_seed42_steps300.pt --checkpoint inbatch=experiments/v4/training/full/dpr_inbatch_k32_b33_seed42_steps300.pt --max-samples 1000 --batch-size 16 --save-query-ranks --output experiments/v4/evaluation/validation_1000/seed42_1000.csv


Device: cuda


Validation pairs: 1000



===== WARMUP =====


I0000 00:00:1792936824.667553   10605 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792936824.739705   10605 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


...

Similarity matrix shape: (2000, 2000)
recall@1: 0.7195
recall@5: 0.8850
recall@20: 0.9575
mrr@10: 0.7924

===== MIXED =====


Similarity matrix shape: (2000, 2000)
recall@1: 0.7465
recall@5: 0.9120
recall@20: 0.9725
mrr@10: 0.8188

===== INBATCH =====


Similarity matrix shape: (2000, 2000)
recall@1: 0.5950
recall@5: 0.8515
recall@20: 0.9575
mrr@10: 0.7054

Results saved to experiments/v4/evaluation/validation_2000/seed42_2000.csv
Per-query ranks saved to experiments/v4/evaluation/validation_2000/seed42_2000_query_ranks.csv
Metadata saved to experiments/v4/evaluation/validation_2000/seed42_2000_metadata.json


PERSISTED: 04_evaluation_2000 at /content/dpr_v4_a100_seed42/04_evaluation_2000
VALIDATION SIZE 2000


,strategy,checkpoint,recall@1,recall@5,recall@20,mrr@10
0,warmup,experiments/v4/warmup/full/dpr_warmup_n10000_b...,0.6270,0.8805,0.9625,0.735692
1,random,experiments/v4/training/full/dpr_random_k4_b33...,0.4925,0.7995,0.9405,0.621462
2,hard,experiments/v4/training/full/dpr_hard_k4_b33_s...,0.7195,0.8850,0.9575,0.792376
3,mixed,experiments/v4/training/full/dpr_mixed_k4_b33_...,0.7465,0.9120,0.9725,0.818837
4,inbatch,experiments/v4/training/full/dpr_inbatch_k32_b...,0.5950,0.8515,0.9575,0.705393


## 10. Summarize compute and package the results

In [ ]:
rows=[]
for name in ['random','hard','mixed','inbatch']:
    k=32 if name=='inbatch' else 4
    namebase=f'dpr_{name}_k{k}_b33_seed42_steps300'
    path=REPO/TRAIN_DIR/(namebase+'.json')
    m=json.loads(path.read_text())
    rows.append({key:m.get(key) for key in ['strategy','optimizer_steps','query_exposures',
             'passage_encoder_input_count','train_seconds','cuda_peak_allocated_gb','mean_train_loss']})
summary=DRIVE_ROOT/'v4_training_summary.csv'
pd.DataFrame(rows).to_csv(summary,index=False)
display(pd.DataFrame(rows))
print('Google Drive experiment folder:',DRIVE_ROOT)
print('Metrics summary:',summary)
print('All completed artifacts are under this persistent Drive folder.')


,strategy,optimizer_steps,query_exposures,passage_encoder_input_count,train_seconds,cuda_peak_allocated_gb,mean_train_loss
0,random,300,9900,49500,172.125,11.512,0.010385
1,hard,300,9900,49500,171.033,11.512,0.837345
2,mixed,300,9900,49500,171.107,11.511,0.861543
3,inbatch,300,9900,9900,42.551,3.914,0.039234


Google Drive experiment folder: /content/dpr_v4_a100_seed42
Metrics summary: /content/dpr_v4_a100_seed42/v4_training_summary.csv
All completed artifacts are under this persistent Drive folder.
